# OSC variant vcf scoring

In [2]:
%%bash
module load miniconda3/24.1.2-py310
module load cuda/12.4.1
conda activate py311

In [9]:
from alphagenome_research.model import dna_model
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.93'

import matplotlib.pyplot as plt
import pandas as pd
from pysam import VariantFile
from io import StringIO
from tqdm import tqdm
import os
# from dotenv import load_dotenv

pd.set_option('display.max_columns', None)


In [10]:
# load_dotenv(override=True)
api_key = os.getenv("AG_API_KEY")

# dna_model = dna_client.create(api_key)
# print(os.getenv('AG_API_KEY'))
dna_model = dna_model.create_from_huggingface('all_folds', 
                                              organism_settings={ 
                                                                 dna_model.Organism.HOMO_SAPIENS: 
                                                                     dna_model.OrganismSettings( 
                                                                                                fasta_path='/users/PAS2905/coraalbers/ag/hg38.fa' ), 
                                                                     # Mandatory dummy entry so the model knows how to load the weights 
                                                                     dna_model.Organism.MUS_MUSCULUS: dna_model.OrganismSettings() 
                                                                     } 
                                              )


Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 11781.75it/s]


In [14]:

# Open local or compressed VCF file
# vcf_in = VariantFile("/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/clinvar.vcf.gz") 


In [4]:
# vcf_in = VariantFile("/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/clinvar.vcf.gz")

# n = 5
# for i, record in enumerate(vcf_in.fetch()):
#     print(record)
#     print(record.info.keys())
#     print(record.info["CLNSIG"])
#     if i + 1 >= n:
#         break

In [5]:

gtf = pd.read_feather(
    '/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/gencode.v46.annotation.gtf.gz.feather'
)

gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol="LMNA")

window_bp = 1_000_000

search_interval = gene_interval.pad(window_bp, window_bp)
# or manually:
# search_start = gene_interval.start - window_bp
# search_end = gene_interval.end + window_bp


In [4]:
gene_symbol = "LMNA"
window_bp = 1_000_000

gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene_symbol)
region = gene_interval.pad(window_bp, window_bp)
vcf_contig = region.chromosome.removeprefix("chr")

vcf_path = "/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/clinvar.vcf.gz"
with VariantFile(vcf_path) as vcf_in:
    nearby = [
        rec for rec in vcf_in.fetch(
            vcf_contig,
            max(region.start - 1, 0),
            region.end,
        )
    ]
print(f"{len(nearby)} variants within ±{window_bp:,} bp of {gene_symbol}")
print(f"Region: {region.chromosome}:{region.start}-{region.end}")

13076 variants within ±1,000,000 bp of LMNA
Region: chr1:155082571-157140081


In [5]:
def variants_near_gene(vcf_path, gene_symbol, gtf, window_bp=1_000_000):
    region = gene_interval.pad(window_bp, window_bp)
    contig = to_vcf_contig(region.chromosome)
    gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene_symbol)
    with VariantFile(vcf_path) as vcf_in:
        # nearby = [
        #     rec for rec in vcf_in.fetch(
        #         vcf_contig,
        #         max(region.start - 1, 0),
        #         region.end,
        #     )
        # ]
        # print(f"{len(nearby)} variants within ±{window_bp:,} bp of {gene_symbol}")
        # print(f"Region: {region.chromosome}:{region.start}-{region.end}")

        for record in vcf_in.fetch(contig, max(region.start - 1, 0), region.end):
            yield record
        
    

In [6]:
variants_near_gene(vcf_path, 'LMNA', gtf, 1_000_000)

<generator object variants_near_gene at 0x16d10ab00>

In [7]:
def to_vcf_contig(chrom: str) -> str:
    return chrom.removeprefix("chr")  # 'chr1' -> '1'

def variants_near_gene(vcf_in, gene_interval, window_bp=1_000_000):
    region = gene_interval.pad(window_bp, window_bp)
    contig = to_vcf_contig(region.chromosome)
    for record in vcf_in.fetch(contig, max(region.start - 1, 0), region.end):
        yield record

# # write a filtered vcf file with variants near the gene
# with VariantFile(vcf_path) as vcf_in, VariantFile("lmna_within_1000000.vcf", "w", header=vcf_in.header) as vcf_out:
#     for record in variants_near_gene(vcf_in, gene_interval):
#         vcf_out.write(record)

In [8]:
PATHOGENIC = {
    "Pathogenic",
    "Likely_pathogenic",
    "Pathogenic/Likely_pathogenic",
    "Uncertain_significance"
    
}

def is_pathogenic(record, gene_filter=False):
    clnsig = record.info.get("CLNSIG")
    if clnsig is None:
        return False
    # pysam returns a tuple for Number=. fields
    if gene_filter:
        if "LMNA:" in record.info.get("GENEINFO", ""):
            if isinstance(clnsig, tuple):
                return any(sig in PATHOGENIC for sig in clnsig)

    else: 
        if isinstance(clnsig, tuple):
            return any(sig in PATHOGENIC for sig in clnsig)
        
    return clnsig in PATHOGENIC




In [9]:
nearby_vcf = '/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_within_1000000.vcf'

with VariantFile(nearby_vcf) as nearby_in, \
     VariantFile("/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_pathogenic_notLMNAgenelabeled.vcf", "w", header=nearby_in.header) as vcf_out:
    count = 0
    for record in nearby_in.fetch():
        if is_pathogenic(record):
            vcf_out.write(record)
            count += 1
print(f"Wrote {count} pathogenic variants")

Wrote 8722 pathogenic variants


## convert vcf to csv

In [10]:
def get_vcf_names(vcf_path):
    """Finds the true column header line inside the VCF file."""
    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#CHROM"):
                # Remove the leading '#' and split by tabs
                return line.strip("#").strip().split("\t")
    raise ValueError("No header line starting with '#CHROM' found.")


vcf_filename = "/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_pathogenic_notLMNAgenelabeled.vcf"
csv_filename = "/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_pathogenic_notLMNAgenelabeled.csv"

# Fetch headers dynamically
column_names = get_vcf_names(vcf_filename)

# Read the VCF file, skipping metadata rows starting with '##'
df = pd.read_csv(
    vcf_filename,
    comment="#",
    sep="\t",
    names=column_names,
    header=None,
    low_memory=False,
)
df['CHROM'] = 'chr1'
df

# Export the clean structure to a CSV
df.to_csv(csv_filename, index=False)
print(f"Successfully exported genomic VCF data to {csv_filename}!")


Successfully exported genomic VCF data to /Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_pathogenic_notLMNAgenelabeled.csv!


## predict variants
from https://www.alphagenomedocs.com/colabs/batch_variant_scoring.html 

In [22]:
csv_file = "/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_pathogenic_notLMNAgenelabeled.csv"
vcf = pd.read_csv(csv_file, sep=',')
print(vcf.columns)
required_columns = ['ID', 'CHROM', 'POS', 'REF', 'ALT']
for column in required_columns:
  if column not in vcf.columns:
    raise ValueError(f'VCF file is missing required column: {column}.')

organism = 'human'  # @param ["human", "mouse"] {type:"string"}

# @markdown Specify length of sequence around variants to predict:
sequence_length = '100KB'  # @param ["16KB", "100KB", "500KB", "1MB"] { type:"string" }
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS[
    f'SEQUENCE_LENGTH_{sequence_length}'
]


# @markdown Specify which scorers to use to score your variants:
score_rna_seq = True  # @param { type: "boolean"}
score_cage = True  # @param { type: "boolean" }
score_procap = True  # @param { type: "boolean" }
score_atac = True  # @param { type: "boolean" }
score_dnase = True  # @param { type: "boolean" }
score_chip_histone = True  # @param { type: "boolean" }
score_chip_tf = True  # @param { type: "boolean" }
score_polyadenylation = True  # @param { type: "boolean" }
score_splice_sites = True  # @param { type: "boolean" }
score_splice_site_usage = True  # @param { type: "boolean" }
score_splice_junctions = True  # @param { type: "boolean" }



# Parse organism specification.
organism_map = {
    'human': dna_client.Organism.HOMO_SAPIENS,
    'mouse': dna_client.Organism.MUS_MUSCULUS,
}
organism = organism_map[organism]

# Parse scorer specification.
scorer_selections = {
    'rna_seq': score_rna_seq,
    'cage': score_cage,
    'procap': score_procap,
    'atac': score_atac,
    'dnase': score_dnase,
    'chip_histone': score_chip_histone,
    'chip_tf': score_chip_tf,
    'polyadenylation': score_polyadenylation,
    'splice_sites': score_splice_sites,
    'splice_site_usage': score_splice_site_usage,
    'splice_junctions': score_splice_junctions,
}

all_scorers = variant_scorers.RECOMMENDED_VARIANT_SCORERS
selected_scorers = [
    all_scorers[key]
    for key in all_scorers
    if scorer_selections.get(key.lower(), False)
]

# Remove any scorers or output types that are not supported for the chosen organism.
unsupported_scorers = [
    scorer
    for scorer in selected_scorers
    if (
        organism.value
        not in variant_scorers.SUPPORTED_ORGANISMS[scorer.base_variant_scorer]
    )
    | (
        (scorer.requested_output == dna_client.OutputType.PROCAP)
        & (organism == dna_client.Organism.MUS_MUSCULUS)
    )
]
if len(unsupported_scorers) > 0:
  print(
      f'Excluding {unsupported_scorers} scorers as they are not supported for'
      f' {organism}.'
  )
  for unsupported_scorer in unsupported_scorers:
    selected_scorers.remove(unsupported_scorer)


# Score variants in the VCF file.
results = []

for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
  variant = genome.Variant(
      chromosome=str(vcf_row.CHROM),
      position=int(vcf_row.POS),
      reference_bases=vcf_row.REF,
      alternate_bases=vcf_row.ALT,
      name=vcf_row.ID,
  )
  interval = variant.reference_interval.resize(sequence_length)

  variant_scores = dna_model.score_variant(
      interval=interval,
      variant=variant,
      variant_scorers=selected_scorers,
      organism=organism,
  )
  results.append(variant_scores)

df_scores = variant_scorers.tidy_scores(results)


# @markdown Other settings:
download_predictions = True  # @param { type: "boolean" }

if download_predictions:
  df_scores.to_csv("/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores_notLMNAgenelabeled.csv", index=False)
#   files.download('variant_scores.csv')

df_scores


Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO'], dtype='object')


100%|██████████| 8722/8722 [2:00:08<00:00,  1.21it/s]  


: 

In [ ]:
# df_scores.to_pickle('lmna_pathogenic_variant_scores.pkl')
# # @markdown Other settings:
# download_predictions = True  # @param { type: "boolean" }

# if download_predictions:
#   df_scores.to_csv('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores.csv', index=False)
#   # files.download('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores.csv')

In [ ]:
columns = [c for c in df_scores.columns if c != 'ontology_curie']
heart_df_scores = df_scores[(df_scores['ontology_curie'] == 'UBERON:0000948')][columns] # heart uberon identifier
heart_df_scores.to_csv('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores_heart_UBERON0000948.csv', index=False)

In [18]:
heart_df_scores.head()

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,track_name,track_strand,Assay title,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,transcription_factor,histone_mark,gtex_tissue,raw_score,quantile_score
387,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0000948 DNase-seq,.,DNase-seq,heart,tissue,embryonic,encode,paired,False,NaN,NaN,NaN,0.141282,0.963955
2883,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K27me3,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K27me3,NaN,-0.177345,-0.999621
2884,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K4me1,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K4me1,NaN,-0.000861,-0.134700
2885,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K4me3,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K4me3,NaN,0.088124,0.988467
2886,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K9ac,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K9ac,NaN,0.101405,0.989576
